# カリキュラム学習チェックポイント別：3D Haarフィルタ／サブバンド挙動の可視化

元ノートブック `256_128model_Train-Copy1.ipynb` は変更せず、このノートブックだけで観察を行います。

## 観察する内容

1. Analysis / Synthesis の8個の3D Haarカーネル
2. 入力画像を分解した `LLL, LLH, LHL, LHH, HLL, HLH, HHL, HHH`
3. 各サブバンドを単独でSynthesisしたときの画像への寄与
4. 2,000 / 24,000 / 50,000 / 80,000 epochでのDVF・再構成結果
5. チェックポイントごとの画像RMSE、DVF RMSE、各バンドRMSE

**注意:** 8バンドは固定Haarフィルタであり、チェックポイントでフィルタ係数自体は変化しません。学習で変化するのは、8バンドを入力としてDVFを予測するモデルです。

In [ ]:
# 必要ライブラリとパス設定
import os, re, math, csv
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

os.environ['VXM_BACKEND'] = 'pytorch'
import voxelmorph as vxm

SAITO_DIR = Path(r'C:\Users\ri0151fv\Saito')
DATA_PATH = SAITO_DIR / 'Data' / 'TrainData_NoBed.npz'
CHECKPOINT_DIR = SAITO_DIR / 'curriculum_checkpoints'
OUTPUT_DIR = SAITO_DIR / 'curriculum_filter_observation'
OUTPUT_DIR.mkdir(exist_ok=True)

# 比較したいepoch。存在しないものは自動的にスキップします。
SELECTED_EPOCHS = [2000, 24000, 50000, 80000]
PATIENT_ID = 0
SHIFT_PIXELS_X = 20.0       # フル解像度での人工的なx方向変位
SLICE_INDEX = None          # Noneなら中央断面
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
print('data:', DATA_PATH)
print('checkpoint dir:', CHECKPOINT_DIR)

In [ ]:
# 利用可能なチェックポイントを確認
def epoch_from_path(path):
    match = re.search(r'epoch_(\d+)\.pth$', path.name)
    return int(match.group(1)) if match else None

available = {epoch_from_path(p): p for p in CHECKPOINT_DIR.glob('pretrain_epoch_*.pth')}
available = {e: p for e, p in available.items() if e is not None}
selected = [(e, available[e]) for e in SELECTED_EPOCHS if e in available]

print('available epochs:', sorted(available))
print('selected:', [e for e, _ in selected])
missing = sorted(set(SELECTED_EPOCHS) - set(available))
if missing:
    print('見つからないためスキップ:', missing)
if not selected:
    raise FileNotFoundError('比較対象のチェックポイントがありません。SELECTED_EPOCHSを変更してください。')

## 3D Haar Analysis / Synthesis

1次元フィルタは次の2つです。

\[
h_L=\frac{1}{\sqrt{2}}[1,1],\qquad
h_H=\frac{1}{\sqrt{2}}[1,-1]
\]

これを z・y・x の3軸で外積し、8個の \(2\times2\times2\) カーネルを作ります。元コードと同じ順序・padding・サンプリングを使用します。

In [ ]:
BAND_NAMES = ['LLL', 'LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH']

def make_3d_filter(fz, fy, fx):
    return fz[:, None, None] * fy[None, :, None] * fx[None, None, :]

class Haar3DAnalysisOnly(nn.Module):
    def __init__(self):
        super().__init__()
        hL = torch.tensor([1., 1.]) / math.sqrt(2.)
        hH = torch.tensor([1., -1.]) / math.sqrt(2.)
        filters, names = [], []
        for zn, zf in zip(['L','H'], [hL,hH]):
            for yn, yf in zip(['L','H'], [hL,hH]):
                for xn, xf in zip(['L','H'], [hL,hH]):
                    filters.append(make_3d_filter(zf, yf, xf))
                    names.append(zn + yn + xn)
        self.register_buffer('weight', torch.stack(filters).unsqueeze(1))
        self.names = names

    def forward(self, x):
        # 元コードと同じく各軸の末尾に1 voxel追加
        return F.conv3d(F.pad(x, (0,1,0,1,0,1)), self.weight)

def down_sampling_3d(w):
    return w[:, :, ::2, ::2, ::2]

def up_sampling_3d(w):
    b, c, d, h, width = w.shape
    out = torch.zeros(b, c, d*2, h*2, width*2, dtype=w.dtype, device=w.device)
    out[:, :, ::2, ::2, ::2] = w
    return out

def create_synthesis_filters(device):
    low = torch.tensor([1.,1.], device=device) / math.sqrt(2.)
    high = torch.tensor([1.,-1.], device=device) / math.sqrt(2.)
    filters = torch.stack([
        make_3d_filter(low,low,low), make_3d_filter(low,low,high),
        make_3d_filter(low,high,low), make_3d_filter(low,high,high),
        make_3d_filter(high,low,low), make_3d_filter(high,low,high),
        make_3d_filter(high,high,low), make_3d_filter(high,high,high),
    ])
    return torch.flip(filters, dims=[1,2,3]).unsqueeze(1)

def synthesis_filter_3d(w_up, filters):
    b, c, d, h, width = w_up.shape
    contributions = []
    for i in range(c):
        y = F.conv3d(w_up[:, i:i+1], filters[i:i+1], padding=1)
        contributions.append(y[:, :, :d, :h, :width])
    contributions = torch.cat(contributions, dim=1)
    return contributions.sum(dim=1, keepdim=True), contributions

analysis = Haar3DAnalysisOnly().to(DEVICE)
synthesis_filters = create_synthesis_filters(DEVICE)
assert analysis.names == BAND_NAMES
print('band order:', analysis.names)

In [ ]:
# 各2x2x2カーネルを、z=0 / z=1の2面に分けて表示
fig, axes = plt.subplots(4, 4, figsize=(10, 10), constrained_layout=True)
for band, name in enumerate(BAND_NAMES):
    kernel = analysis.weight[band, 0].detach().cpu().numpy()
    for z in range(2):
        ax = axes[band // 2, (band % 2) * 2 + z]
        im = ax.imshow(kernel[z], cmap='coolwarm', vmin=-kernel.max(), vmax=kernel.max())
        ax.set_title(f'{name}: z={z}')
        for y in range(2):
            for x in range(2):
                ax.text(x, y, f'{kernel[z,y,x]:+.3f}', ha='center', va='center')
        ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('3D Haar Analysis kernels')
fig.savefig(OUTPUT_DIR / 'haar_3d_kernels.png', dpi=180, bbox_inches='tight')
plt.show()

## 入力画像に実際にフィルタを適用した結果

- `LLL`: 3方向とも低周波で、画像の大まかな形・濃度を主に保持します。
- L/Hが混ざるバンド: 該当方向の境界・変化を強く表します。
- `HHH`: 3方向すべての急な変化を表し、値は一般に小さくなります。

ここから表示する画像は模式図ではなく、`TrainData_NoBed.npz` のCTへ元コードと同じフィルタを実際に適用したテンソルです。

高周波成分には正負の値があるため、**0を中間灰色、負値を黒、正値を白**として表示します。バンド間で振幅が大きく違うため、表示範囲は各バンドの1–99 percentileで個別調整します。画像同士の絶対振幅比較には、下のenergy表を使用してください。

In [ ]:
# データ読込と固定観察画像（全25 GiBをRAMへ展開しない省メモリ版）
import zipfile
import struct

def load_one_patient_from_stored_npz(npz_path, array_name, patient_id):
    # このデータはNPZ内で非圧縮（ZIP_STORED）のため、内部の.npyを直接memmapできます。
    member_name = array_name + '.npy'
    with zipfile.ZipFile(npz_path, 'r') as archive:
        info = archive.getinfo(member_name)
        if info.compress_type != zipfile.ZIP_STORED:
            raise ValueError('圧縮NPZでは直接memmapできません。非圧縮NPZへ変換してください。')

        # ZIPローカルヘッダから.npy本体の開始位置を求める
        with open(npz_path, 'rb') as raw_file:
            raw_file.seek(info.header_offset)
            local_header = raw_file.read(30)
            fields = struct.unpack('<IHHHHHIIIHH', local_header)
            filename_length, extra_length = fields[-2], fields[-1]
            npy_start = info.header_offset + 30 + filename_length + extra_length

            raw_file.seek(npy_start)
            version = np.lib.format.read_magic(raw_file)
            if version == (1, 0):
                shape, fortran_order, dtype = np.lib.format.read_array_header_1_0(raw_file)
            else:
                shape, fortran_order, dtype = np.lib.format.read_array_header_2_0(raw_file)
            array_start = raw_file.tell()

    if fortran_order:
        raise ValueError('Fortran-order配列には未対応です。')
    if not (0 <= patient_id < shape[-1]):
        raise IndexError(f'PATIENT_ID={patient_id} は範囲外です。患者数={shape[-1]}')

    mapped = np.memmap(
        npz_path, mode='r', dtype=dtype, offset=array_start,
        shape=shape, order='C'
    )
    # 元配列は (D,H,W,患者)。指定患者だけコピーしfloat32へ変換する。
    patient = np.asarray(mapped[..., patient_id], dtype=np.float32)
    del mapped
    return patient

patient_volume = load_one_patient_from_stored_npz(DATA_PATH, 'Train', PATIENT_ID)
print('loaded one patient:', patient_volume.shape, patient_volume.dtype,
      f'{patient_volume.nbytes / 1024**2:.1f} MiB')
moving = torch.from_numpy(patient_volume[None, None]).to(DEVICE)
slice_full = moving.shape[2] // 2 if SLICE_INDEX is None else SLICE_INDEX

with torch.no_grad():
    analyzed = analysis(moving)
    bands = down_sampling_3d(analyzed)
    upsampled = up_sampling_3d(bands)
    reconstructed, contributions = synthesis_filter_3d(upsampled, synthesis_filters)

slice_low = bands.shape[2] // 2
slice_analysis = analyzed.shape[2] // 2

def show_real_subbands(tensor, slice_index, stage, filename):
    # 実テンソル8成分を、符号を保持した白黒画像として表示する。
    fig, axes = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)
    for i, (ax, name) in enumerate(zip(axes.flat, BAND_NAMES)):
        image = tensor[0, i, slice_index].detach().cpu().numpy()
        if name == 'LLL':
            lo, hi = np.percentile(image, [1, 99])
        else:
            lim = max(abs(np.percentile(image, 1)), abs(np.percentile(image, 99)), 1e-8)
            lo, hi = -lim, lim
        ax.imshow(image, cmap='gray', vmin=lo, vmax=hi)
        ax.set_title(f'{name} {stage}')
        ax.axis('off')
    fig.suptitle(f'Moving — {stage}', fontsize=18)
    fig.savefig(OUTPUT_DIR / filename, dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()

# 1. Analysis filter直後（まだ間引いていない）
show_real_subbands(analyzed, slice_analysis, 'analysis', '01_analysis_filters_grayscale.png')

# 2. Downsampling直後（モデルへ入力される8成分）
show_real_subbands(bands, slice_low, 'downsampled', '02_downsampled_grayscale.png')

# 3. Upsampling直後（偶数位置以外は0）
show_real_subbands(upsampled, slice_full, 'upsampled', '03_upsampled_grayscale.png')

# energyを数値でも確認
fig, axes = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)
energy_rows = []
for i, (ax, name) in enumerate(zip(axes.flat, BAND_NAMES)):
    image = bands[0, i, slice_low].detach().cpu().numpy()
    lim = max(abs(np.percentile(image, 1)), abs(np.percentile(image, 99)), 1e-8)
    ax.imshow(image, cmap='gray', vmin=-lim, vmax=lim)
    rms = float(torch.sqrt(torch.mean(bands[:, i] ** 2)).item())
    energy = float(torch.sum(bands[:, i] ** 2).item())
    energy_rows.append({'band': name, 'rms': rms, 'energy': energy})
    ax.set_title(f'{name}  RMS={rms:.4g}')
    ax.axis('off')
fig.suptitle('Analysis + downsampling: 8 subbands')
fig.savefig(OUTPUT_DIR / 'analysis_subbands_with_rms.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

print('再構成 MAE:', torch.mean(torch.abs(moving - reconstructed)).item())
print('再構成 最大絶対誤差:', torch.max(torch.abs(moving - reconstructed)).item())
energy_rows

In [ ]:
# 4. 各サブバンドへSynthesis filterを適用した直後の寄与
fig, axes = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)
for i, (ax, name) in enumerate(zip(axes.flat, BAND_NAMES)):
    image = contributions[0, i, slice_full].detach().cpu().numpy()
    if name == 'LLL':
        lo, hi = np.percentile(image, [1, 99])
    else:
        lim = max(abs(np.percentile(image, 1)), abs(np.percentile(image, 99)), 1e-8)
        lo, hi = -lim, lim
    ax.imshow(image, cmap='gray', vmin=lo, vmax=hi)
    ax.set_title(f'{name} synthesis')
    ax.axis('off')
fig.suptitle('Moving — synthesis filter output', fontsize=18)
fig.savefig(OUTPUT_DIR / '04_synthesis_filters_grayscale.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

## チェックポイントごとのモデル読み込みと比較

保存形式は次の辞書です。

```python
{
    "epoch": epoch,
    "model_state_dict": model3D.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "shift_range": shift_range,
}
```

したがって、モデルには `checkpoint['model_state_dict']` を読み込みます。元コードと同じ `VxmDense_128_256_256` と特徴数を使用します。

In [ ]:
NB_FEATURES = [
    [32, 64, 64, 64, 64],
    [64, 64, 64, 64, 64, 32, 16, 16],
]

def make_model():
    return vxm.networks.VxmDense_128_256_256(
        (128, 256, 256), NB_FEATURES, int_steps=0
    ).to(DEVICE)

def load_checkpoint_model(path):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=True)
    model = make_model()
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model, checkpoint

transformer_low = vxm.layers.SpatialTransformer((64, 128, 128)).to(DEVICE)
transformer_full = vxm.layers.SpatialTransformer((128, 256, 256)).to(DEVICE)

# 評価用の既知変位：正解を知っているため、epoch間を公平に比較できます。
true_flow_full = torch.zeros((1, 3, *moving.shape[2:]), dtype=moving.dtype, device=DEVICE)
true_flow_full[:, 2] = SHIFT_PIXELS_X   # channel 0=z, 1=y, 2=x
target = transformer_full(moving, true_flow_full)

with torch.no_grad():
    moving_bands = down_sampling_3d(analysis(moving))
    target_bands = down_sampling_3d(analysis(target))

true_flow_low = torch.zeros((1, 3, *moving_bands.shape[2:]), dtype=moving.dtype, device=DEVICE)
true_flow_low[:, 2] = SHIFT_PIXELS_X / 2.0

In [ ]:
# 全チェックポイントを順に評価（GPUメモリ節約のためモデルは1つずつ破棄）
results = []
band_results = []
snapshots = {}

for epoch, path in selected:
    model, checkpoint = load_checkpoint_model(path)
    with torch.no_grad():
        predicted_flow = model(moving_bands, target_bands)
        warped_bands = torch.cat([
            transformer_low(moving_bands[:, i:i+1], predicted_flow)
            for i in range(8)
        ], dim=1)
        moved, moved_contrib = synthesis_filter_3d(up_sampling_3d(warped_bands), synthesis_filters)

        image_rmse = torch.sqrt(torch.mean((target - moved) ** 2)).item()
        dvf_rmse = torch.sqrt(torch.mean((true_flow_low - predicted_flow) ** 2)).item()
        results.append({
            'epoch': epoch,
            'saved_shift_range': checkpoint.get('shift_range'),
            'image_rmse': image_rmse,
            'dvf_rmse': dvf_rmse,
            'predicted_x_mean': predicted_flow[:, 2].mean().item(),
        })
        for i, name in enumerate(BAND_NAMES):
            rmse = torch.sqrt(torch.mean((target_bands[:, i] - warped_bands[:, i]) ** 2)).item()
            band_results.append({'epoch': epoch, 'band': name, 'band_rmse': rmse})
        snapshots[epoch] = {
            'moved': moved.detach().cpu(),
            'flow': predicted_flow.detach().cpu(),
            'warped_bands': warped_bands.detach().cpu(),
            'contributions': moved_contrib.detach().cpu(),
        }
    del model, predicted_flow, warped_bands, moved, moved_contrib
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results

In [ ]:
# epochごとの再構成画像とx-DVF
n = len(selected)
fig, axes = plt.subplots(n, 3, figsize=(14, 4*n), squeeze=False, constrained_layout=True)
target_np = target[0, 0, slice_full].detach().cpu().numpy()
vmin, vmax = np.percentile(target_np, [1,99])
for row, (epoch, _) in enumerate(selected):
    result = next(r for r in results if r['epoch'] == epoch)
    moved_np = snapshots[epoch]['moved'][0, 0, slice_full].numpy()
    flow_np = snapshots[epoch]['flow'][0, 2, slice_low].numpy()
    error_np = np.abs(target_np - moved_np)
    axes[row,0].imshow(moved_np, cmap='gray', vmin=vmin, vmax=vmax)
    axes[row,0].set_title(f'Epoch {epoch}: reconstructed\nRMSE={result["image_rmse"]:.5f}')
    im1 = axes[row,1].imshow(flow_np, cmap='coolwarm')
    axes[row,1].set_title(f'Predicted x-DVF (truth={SHIFT_PIXELS_X/2:.1f})\nmean={result["predicted_x_mean"]:.3f}')
    fig.colorbar(im1, ax=axes[row,1], fraction=.046, pad=.04)
    im2 = axes[row,2].imshow(error_np, cmap='inferno')
    axes[row,2].set_title('Absolute image error')
    fig.colorbar(im2, ax=axes[row,2], fraction=.046, pad=.04)
    for ax in axes[row]: ax.axis('off')
fig.savefig(OUTPUT_DIR / 'checkpoint_reconstruction_comparison.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# 各チェックポイント × 各サブバンドのワープ後画像
for epoch, _ in selected:
    wb = snapshots[epoch]['warped_bands']
    fig, axes = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)
    for i, (ax, name) in enumerate(zip(axes.flat, BAND_NAMES)):
        image = wb[0, i, slice_low].numpy()
        lim = max(abs(np.percentile(image,1)), abs(np.percentile(image,99)), 1e-8)
        rmse = next(r['band_rmse'] for r in band_results if r['epoch']==epoch and r['band']==name)
        ax.imshow(image, cmap='gray', vmin=-lim, vmax=lim)
        ax.set_title(f'{name} warped\nRMSE={rmse:.4g}')
        ax.axis('off')
    fig.suptitle(f'Warped subbands — epoch {epoch}')
    fig.savefig(OUTPUT_DIR / f'warped_subbands_epoch_{epoch:05d}.png', dpi=180, bbox_inches='tight')
    plt.show()

In [ ]:
# 発表用：最終選択チェックポイントの Moving / Target / Warped（白黒）
final_epoch = selected[-1][0]
moving_np = moving[0, 0, slice_full].detach().cpu().numpy()
target_np = target[0, 0, slice_full].detach().cpu().numpy()
warped_np = snapshots[final_epoch]['moved'][0, 0, slice_full].numpy()
display_values = np.concatenate([moving_np.ravel(), target_np.ravel()])
vmin, vmax = np.percentile(display_values, [1, 99])

fig, axes = plt.subplots(1, 3, figsize=(16, 5.4), constrained_layout=True)
for ax, image, title in zip(
    axes, [moving_np, target_np, warped_np],
    ['Moving', 'Target (known shift)', f'Warped — epoch {final_epoch}']
):
    ax.imshow(image, cmap='gray', vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=18)
    ax.axis('off')
fig.savefig(OUTPUT_DIR / '05_moving_target_warped_grayscale.png', dpi=200,
            bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# 指標推移とCSV保存
epochs = [r['epoch'] for r in results]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(epochs, [r['image_rmse'] for r in results], marker='o')
axes[0].set(title='Image RMSE', xlabel='Epoch', ylabel='RMSE'); axes[0].grid(True)
axes[1].plot(epochs, [r['dvf_rmse'] for r in results], marker='o', color='tab:orange')
axes[1].set(title='DVF RMSE', xlabel='Epoch', ylabel='RMSE'); axes[1].grid(True)
fig.savefig(OUTPUT_DIR / 'checkpoint_metric_curves.png', dpi=180, bbox_inches='tight')
plt.show()

with (OUTPUT_DIR / 'checkpoint_metrics.csv').open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader(); writer.writerows(results)
with (OUTPUT_DIR / 'checkpoint_band_metrics.csv').open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=band_results[0].keys())
    writer.writeheader(); writer.writerows(band_results)
print('保存先:', OUTPUT_DIR)

## 結果の読み方

- フィルタ係数とAnalysis結果は全epochで共通です。チェックポイント比較で変化するのは予測DVFと、そのDVFでワープした各サブバンドです。
- `LLL` の誤差は大域的な濃度・形状のずれを反映しやすく、Hを含むバンドの誤差は境界や細部のずれを反映しやすいです。
- epoch増加に伴い画像RMSEとDVF RMSEが下がり、予測x-DVF平均が正解 `SHIFT_PIXELS_X / 2` に近づけば、カリキュラム学習が既知変位を学べていると解釈できます。
- バンドごとの表示範囲は見やすさのため個別正規化しています。強度の比較にはRMS・energy・band RMSEの数値を使ってください。
- 画像RMSEだけが改善してDVF RMSEが改善しない場合、正しい変位ではなく濃度類似性だけを合わせている可能性があります。